In [ ]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memoria inicial: {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        #si es un número (int o float), reducimos su tamaño
        if col_type != object and str(col_type) != 'category' and str(col_type) != 'datetime64[ns]':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                    
        #si es un texto repetitivo (ej. nombre de tienda), lo pasamos a categoría
        elif col_type == object and col != 'date':
            df[col] = df[col].astype('category')
            
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memoria final: {end_mem:.2f} MB. ¡Reducción del {100 * (start_mem - end_mem) / start_mem:.1f}%!')
    return df

In [ ]:
#carga
df_sales = pd.read_csv('sales_train_evaluation.csv')
df_calendar = pd.read_csv('calendar.csv')
df_prices = pd.read_csv('sell_prices.csv')

#transformacion
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
df_ts = pd.melt(df_sales, 
                id_vars=id_vars, 
                var_name='d', 
                value_name='sales')

#downcasting
df_ts = reduce_mem_usage(df_ts)
df_calendar = reduce_mem_usage(df_calendar)
df_prices = reduce_mem_usage(df_prices)

#cruzar con date
cols_calendario = ['d', 'date', 'wm_yr_wk', 'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']
df_final = pd.merge(df_ts, df_calendar[cols_calendario], on='d', how='left')
df_final['date'] = pd.to_datetime(df_final['date'])

#cruzar con precios
df_final = pd.merge(df_final, df_prices, 
                    on=['store_id', 'item_id', 'wm_yr_wk'], 
                    how='left')

#liberar RAM
del df_sales, df_ts, df_calendar, df_prices
import gc
gc.collect()

df_final.info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

#serie temporal indexada por la decha
ts_data = df_final.groupby('date')['sales'].sum()

#descomposición matemática (semanal)
descomposicion = seasonal_decompose(ts_data, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
descomposicion.observed.plot(ax=axes[0], color='#2c3e50')
axes[0].set_ylabel('Observado')
axes[0].set_title('Descomposición de la Serie Temporal (Ventas Totales)', fontweight='bold')

descomposicion.trend.plot(ax=axes[1], color='#e74c3c')
axes[1].set_ylabel('Tendencia')

descomposicion.seasonal.plot(ax=axes[2], color='#27ae60')
axes[2].set_ylabel('Estacionalidad')

descomposicion.resid.plot(ax=axes[3], color='#8e44ad', style='.')
axes[3].set_ylabel('Ruido / Residuos')
plt.tight_layout()
plt.show()

#estacionalidad semanal
df_final['day_of_week'] = df_final['date'].dt.dayofweek
dias_semana = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df_final['day_name'] = df_final['day_of_week'].map(dias_semana)

#media de ventas por dia
ventas_por_dia = df_final.groupby('day_name')['sales'].mean().reindex(dias_semana.values())

plt.figure(figsize=(10, 5))
sns.barplot(x=ventas_por_dia.index, y=ventas_por_dia.values, palette="viridis")
plt.title('Media de Ventas por Día de la Semana', fontsize=15, fontweight='bold')
plt.ylabel('Media de Unidades Vendidas')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#impacto de variables exogenas
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_final, x='snap_CA', y='sales', showfliers=False, palette="Set2")
plt.title('Impacto de las Ayudas SNAP en las Ventas (California)', fontsize=15, fontweight='bold')
plt.xticks([0, 1], ['Día Normal (SNAP=0)', 'Día con Ayuda (SNAP=1)'])
plt.ylabel('Distribución de Ventas')
plt.show()

In [ ]:
#analisis productos estrella
top_10_ids = df_final.groupby('item_id')['sales'].sum().nlargest(10).index.tolist()

#filtramos
df_top10 = df_final[df_final['item_id'].isin(top_10_ids)]

#grafica 1: evolucion temporal individual
top_5_ids = top_10_ids[:5]
df_top5 = df_top10[df_top10['item_id'].isin(top_5_ids)]

#agrupado semanalmente
df_top5_weekly = df_top5.groupby(['item_id', pd.Grouper(key='date', freq='W')])['sales'].sum().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=df_top5_weekly, x='date', y='sales', hue='item_id', palette='tab10', linewidth=2)
plt.title('Evolución Semanal de los 5 Productos Top Ventas', fontsize=16, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Unidades Vendidas (Semanales)')
plt.legend(title='Producto ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


#grafica 2: matriz de correlacion
df_pivot = df_top10.pivot_table(index='date', columns='item_id', values='sales', aggfunc='sum')
matriz_correlacion = df_pivot.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_correlacion, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación entre los 10 Productos Estrella', fontsize=15, fontweight='bold')
plt.xlabel('Producto')
plt.ylabel('Producto')
plt.tight_layout()
plt.show()